# 01 · LLM Orchestrated Agent Workflow
Conversation Observer가 최근 대화에서 현재 요청과 관련된 사실만 관찰해 전달하고, Cohere Rerank는 현재 요청을 Chat 또는 Work로 판별합니다. Work는 오케스트레이터가 전문 Agent 협업을 조율하고 Synthesis Agent가 결과를 통합한 뒤, Evaluation Middleware가 결과와 Tool audit을 평가해 필요할 때만 보완합니다.

**학습 목표:** 공개 코어 API `inspect_route()`와 `run_workflow()`를 호출해 route·오케스트레이션·Tool audit·평가/보완 trace를 관찰합니다.  
**실행 경계:** 외부 비즈니스 시스템에는 쓰지 않습니다. 다만 Marketing Agent는 로컬 HTML/이미지 미리보기, UI는 로컬 debug snapshot을 만들 수 있습니다.  

**환경 변수:** UI와 코어 workflow 모두 실제 Cohere Rerank 및 AI Core LLM을 사용하므로 루트 `.env` 설정이 필요합니다.

In [8]:
from pathlib import Path
import sys
from IPython.display import HTML, display

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.agent_workflow.ui import WORKFLOW_GRAPH_SVG

display(HTML(WORKFLOW_GRAPH_SVG))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Agent Workflow Chat</h2>
<p style="margin: 7px 0;">Agent graph와 채팅 UI에서 요청을 보냅니다. Work는 전문 Agent·Synthesis Agent 실행 뒤 Evaluation Middleware가 Tool audit과 결과를 평가합니다. 보완은 필요할 때만 최대 3회 실행되며, Tool은 읽기 전용입니다.</p>
  </div>
</div>


In [1]:
import importlib
import src.agent_workflow.ui as workflow_ui_module

workflow_ui_module = importlib.reload(workflow_ui_module)
if 'workflow_ui' in globals():
    workflow_ui.close()
workflow_ui = workflow_ui_module.launch_workflow_ui()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 Chat/Work 분기 뒤 Specialist·Synthesis·Evaluation Middleware의 승인 또는 보완 흐름을 확인한다.</li>
<li>UI가 실행되지 않으면 AI Core 환경 설정, Cohere Rerank deployment와 모델 이름을 먼저 점검한다.</li>
<li>아래 선택 심화에서는 route-only 비교와 실행 계획·Tool audit·평가 결과를 확인한다.</li>
</ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI와 같은 공개 코어 API만 호출합니다. 일반 실행은 AI Core와 Cohere Rerank를 사용하므로 비용이 발생할 수 있습니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">준비: Chat/Work router와 오케스트레이션</h2>
<p style="margin: 7px 0;">Conversation Observer가 최근 대화에서 관련 사실만 전달하고, Router가 Chat 또는 Work를 판별합니다. Work는 Specialist·Synthesis 실행 뒤 Evaluation Middleware가 Tool audit과 최종 답변을 평가하며, 보완은 최대 3회입니다.</p>
  </div>
</div>


In [ ]:
# 노트북은 구현을 복사하지 않고 공개 코어 API만 사용합니다.
from src.agent_workflow.food_agents import list_food_agents
from src.agent_workflow.lab_core import (
    inspect_route,
    run_workflow,
    run_workflow_steps,
)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Food Agent 설정 보기</h2>
<p style="margin: 7px 0;">공개 API가 반환한 6개 Agent의 설명, System Prompt, Tool 종류와 실행 경계를 확인합니다. Tool은 계산·검증·읽기 전용 API·Agent-as-a-Tool입니다. 외부 비즈니스 시스템에는 쓰지 않지만, Marketing Tool은 로컬 미리보기 HTML/이미지 asset을 만들 수 있습니다.</p>
  </div>
</div>


In [ ]:
agent_catalog = list_food_agents()
for agent in agent_catalog:
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print("\n[System Prompt]")
    print(agent['system_prompt'])
    print()



<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Cell 단위 Workflow 실행</h2>
<p style="margin: 7px 0;">아래부터는 Observer, Router, Planning Agent, 실행 계획, 다음 Agent를 각각 별도 셀에서 진행합니다. <code>다음 Agent 또는 평가 1회 진행</code> 셀을 다시 실행할 때마다 정확히 한 Agent 또는 하나의 평가 단위만 진행됩니다.</p>
  </div>
</div>


In [ ]:
request = "재고를 분석해서 발주가 필요한 품목을 알려줘."
workflow_steps = run_workflow_steps(request)
result = None
print('Workflow session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.')


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Conversation Observer</h2><p style="margin: 7px 0;">현재 요청에 이전 대화가 필요한지와 외부 데이터가 필요한지를 한 셀에서 확인합니다.</p></div></div>


In [ ]:
route_step = next(workflow_steps)
assert route_step['step'] == 'route', route_step['step']
print(route_step['decision'])


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Router</h2><p style="margin: 7px 0;">Cohere Rerank의 선택 route, score margin, 저신뢰도 여부를 분리해 봅니다.</p></div></div>


In [ ]:
plan_step = next(workflow_steps)
assert plan_step['step'] == 'plan', plan_step['step']
for number, stage in enumerate(plan_step['plan']['stages'], start=1):
    print(f"{number}. {', '.join(stage['agent_ids'])}: {stage['objective']}")
print('최종 Agent:', plan_step['plan']['final_agent_id'])


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Planning Agent와 실행 계획</h2><p style="margin: 7px 0;">Planning Agent 실행과 LLM이 고른 단계별 Agent 계획을 두 개의 셀로 분리합니다.</p></div></div>


In [ ]:
agent_step = next(workflow_steps)
assert agent_step['step'] == 'agent', agent_step['step']
print(f"{agent_step['agent_label']} ({agent_step['agent_id']})")
print(agent_step['answer'])


In [ ]:
next_step = next(workflow_steps)
print(next_step)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">6. 다음 Agent 또는 평가 1회 진행</h2><p style="margin: 7px 0;">이 셀을 다시 실행할 때마다 다음 Specialist 또는 Synthesis Agent 하나, 혹은 Evaluation Middleware의 한 단위만 진행됩니다. <code>complete</code>가 출력될 때까지 반복하세요.</p></div></div>


In [ ]:
for next_step in workflow_steps:
    if next_step['step'] == 'agent':
        print(f"[실행] {next_step['agent_label']} ({next_step['agent_id']})")
        print(next_step['answer'])
    elif next_step['step'] == 'complete':
        result = next_step['result']
        print(f"[complete] final_agent={result.get('final_agent_id')}")
        print(result['result'])
        break


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 실행 없이 route 비교</h2>
<p style="margin: 7px 0;">여러 문구를 비교할 때는 inspect_route()만 호출합니다. 이 API는 Cohere Rerank만 사용하며 Planning·전문 Agent는 실행하지 않습니다. low_confidence=True이면 실제 workflow는 명확화 요청으로 멈춥니다.</p>
  </div>
</div>


In [ ]:
for request in ["안녕하세요. workflow와 loop의 차이를 설명해줘.", "재고를 분석해서 발주가 필요한 품목을 알려줘."]:
    route = inspect_route(request, min_margin=0.15)
    print(f"{route['selected']:<4} | margin={route['margin']:.3f} | low_confidence={route['low_confidence']} | {request}")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 실행 계획·Tool audit 확인</h2>
<p style="margin: 7px 0;">바로 앞 Work 실행의 반환값에서 오케스트레이션 브리프, 단계별 Agent, Tool audit, Synthesis 결과와 Evaluation Middleware의 승인·보완·최대 횟수 도달 여부를 읽습니다. 명확화 route였다면 Work 실행 정보가 없음을 표시합니다.</p>
  </div>
</div>


In [ ]:
plan = result.get("orchestration_plan")
if not plan:
    print("Work 실행 없음: route=", result["route"])
else:
    print("[Orchestrator 공통 브리프]", plan["brief"])
    for number, stage in enumerate(plan["stages"], start=1):
        print(f"단계 {number}: {stage['objective']} → {', '.join(stage['agent_ids'])}")
    print("최종 담당 Agent:", plan["final_agent_id"])
    for agent_id, agent_result in result["agent_results"].items():
        print(f"{agent_id}: {agent_result['answer'][:100]}")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Conversation Observer 후속 요청</h2>
<p style="margin: 7px 0;">이전 HTML 초안에 대한 수정 요청을 실행하고, Observer가 전체 대화 대신 현재 요청에 필요한 관찰만 전달했는지 반환값으로 확인합니다. 이 셀은 실제 AI Core·Cohere 호출을 수행합니다.</p>
  </div>
</div>


In [ ]:
follow_up_history = [
    {"role": "user", "content": "프로틴 런치 프로모션 HTML을 만들어줘."},
    {"role": "assistant", "content": "프로틴 런치 HTML 초안을 만들었습니다."},
]
follow_up_result = run_workflow(
    "가격을 5,000원으로 바꾸고 문구를 더 간결하게 수정해줘.",
    conversation_history=follow_up_history,
)
print("route: ", follow_up_result["route"])
print("전달한 최근 대화 수: ", len(follow_up_history))
print("최종 응답: ", follow_up_result["result"])


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">상태와 감사 경계</h2>
<p style="margin: 7px 0;">UI는 로컬 파일 <code>4_agent_patterns/artifacts/debug/latest_workflow_conversation.json</code>에 최신 대화 snapshot 하나를 덮어씁니다. 사용자 메시지, 최근 대화 맥락, workflow 요청·trace·Agent 상태·최종 응답을 포함할 수 있지만 Settings와 자격 증명은 저장하지 않습니다. 이 파일은 현재 로컬 사용자 권한으로 접근 가능하며 실행할 때마다 교체됩니다. 실습이 끝나면 조직의 보존 정책에 맞게 내용을 확인하고 필요하면 파일을 삭제하세요.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">실습 미션</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>Chat 예시와 Work 예시를 각각 작성하고, inspect_route() 결과와 기대 route를 비교한다.</li>
<li>최소 margin을 높여 low_confidence가 되면 Agent가 실행되지 않고 명확화 요청으로 끝나는지 확인한다.</li>
<li>허용된 Agent와 Tool만 선택되고, 외부 비즈니스 시스템에는 쓰지 않는다는 실행 경계를 확인한다.</li>
<li>Synthesis 결과 뒤 Evaluation Middleware가 승인하거나 필요한 Agent만 보완하도록 지시하는지 확인한다.</li>
<li>route score·margin·선택 Agent·Tool 이름·평가 이유와 보완 단계가 trace/audit에 남는지 확인한다.</li>
<li>보완 평가는 최대 3회이며, 외부 쓰기 Tool은 실행하지 않는지 확인한다.</li>
</ul>
  </div>
</div>
